# Threshold and Duration Analysis: Heat Stress Exposure Patterns

This notebook analyzes threshold exceedances and duration of heat stress events:

**Analysis Types:**
1. Threshold exceedance frequencies
2. Duration curves and cumulative distributions
3. Percentile analysis (90th, 95th, 99th)
4. Risk matrices (frequency × intensity)
5. Cumulative stress days
6. Heat wave identification and characteristics
7. Recovery period analysis
8. Consecutive day analysis

**Thresholds Analyzed:** Multiple temperature and VPD thresholds representing different stress levels

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Threshold analysis imports complete!")

In [ ]:
# Configuration
DATA_DIR = Path('../..')  # Up two levels from notebooks/03_analysis/ to research/
OUTPUT_DIR = Path('../../figures/threshold')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Import all paths and definitions from centralized config
import sys
sys.path.append(str(DATA_DIR))  # Add research/ to path
from config import (
    FOCUS_STATES, STATE_NAMES, STATE_REGIONS, MASK_FILE,
    PROCESSED_NIGHTTIME_DIR, PROCESSED_DAYTIME_DIR, PROCESSED_VPD_DIR
)

# Use paths from config
NIGHTTIME_DIR = PROCESSED_NIGHTTIME_DIR
DAYTIME_DIR = PROCESSED_DAYTIME_DIR
VPD_DIR = PROCESSED_VPD_DIR

# Heat stress thresholds
TEMP_THRESHOLDS = [25, 27, 30, 32, 35, 37, 40]  # °C
VPD_THRESHOLDS = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]  # kPa

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Mask file: {MASK_FILE}")
print(f"Nighttime data: {NIGHTTIME_DIR}")
print(f"Daytime data: {DAYTIME_DIR}")
print(f"VPD data: {VPD_DIR}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Loaded {len(FOCUS_STATES)} focus states from config.py")

In [ ]:
# Helper functions
def load_monthly_data(data_dir, year_start=2010, year_end=2023):
    """Load monthly data for a given period."""
    files = sorted(data_dir.glob('*.nc'))
    datasets = []
    for f in files:
        year_month = f.stem.split('_')[-1]
        year = int(year_month[:4])
        if year_start <= year <= year_end:
            datasets.append(xr.open_dataset(f))
    if datasets:
        return xr.concat(datasets, dim='time')
    return None

def compute_state_mean(data, state_id, _state_mask=None):
    """Compute spatial mean for a specific state.
    
    Converts timedelta64[ns] to float hours if needed.
    
    Parameters
    ----------
    data : xarray.DataArray
        Data to compute mean for
    state_id : int
        State ID from region_mask
    _state_mask : ignored
        Deprecated parameter for backwards compatibility (state_mask is global)
    """
    mask = state_mask == state_id
    masked_data = data.where(mask)
    state_mean = masked_data.mean(dim=['lat', 'lon'])
    
    # Convert timedelta to hours if data is stored as timedelta
    if state_mean.dtype.kind == 'm':  # 'm' = timedelta
        # Convert nanoseconds to hours: divide by (3600 * 10^9)
        state_mean = state_mean / np.timedelta64(1, 'h')
        state_mean = state_mean.astype(np.float64)
    
    return state_mean

def identify_heatwaves(data, threshold, min_duration=3):
    """Identify heat wave events (consecutive days above threshold)."""
    above_threshold = (data > threshold).astype(int)
    
    # Find consecutive sequences
    events = []
    in_event = False
    event_start = None
    
    for i, val in enumerate(above_threshold.values):
        if val == 1 and not in_event:
            in_event = True
            event_start = i
        elif val == 0 and in_event:
            duration = i - event_start
            if duration >= min_duration:
                events.append({'start': event_start, 'end': i-1, 'duration': duration})
            in_event = False
    
    # Handle case where data ends during an event
    if in_event:
        duration = len(above_threshold) - event_start
        if duration >= min_duration:
            events.append({'start': event_start, 'end': len(above_threshold)-1, 'duration': duration})
    
    return events

print("Helper functions defined!")

In [ ]:
# Load data and mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask
ds_mask.close()

print("Loading datasets (2010-2023)...")
ds_night = load_monthly_data(NIGHTTIME_DIR, 2010, 2023)
ds_day = load_monthly_data(DAYTIME_DIR, 2010, 2023)
ds_vpd = load_monthly_data(VPD_DIR, 2010, 2023)
print("Data loaded!")

## Plot 1: Threshold Exceedance Frequency

How often do different thresholds get exceeded?

In [ ]:
# Compute threshold exceedance frequencies
fig, axes = plt.subplots(4, 4, figsize=(20, 16))  # Updated to 4x4 for 14 states
axes = axes.flatten()

# Focus on summer months
summer_day = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    # Get state data
    state_data = compute_state_mean(summer_day, state_id, state_mask)
    
    # Define thresholds for "hot days" (hours > X)
    threshold_hours = [4, 6, 8, 10, 12]  # hours per day above 30°C
    frequencies = []
    
    for thresh in threshold_hours:
        days_exceeding = np.sum(state_data.values > thresh)
        freq_percent = (days_exceeding / len(state_data)) * 100
        frequencies.append(freq_percent)
    
    # Plot
    bars = ax.bar(range(len(threshold_hours)), frequencies,
                  color=plt.cm.Reds(np.linspace(0.4, 0.9, len(threshold_hours))))
    ax.set_xticks(range(len(threshold_hours)))
    ax.set_xticklabels([f'>{h}h' for h in threshold_hours])
    ax.set_xlabel('Hours Above 30°C per Day', fontsize=9)
    ax.set_ylabel('Frequency (%)', fontsize=9)
    ax.set_title(f'{state_name}\nThreshold Exceedance (Summer 2010-2023)', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, frequencies):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

# Hide unused subplots
for idx in range(len(FOCUS_STATES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_threshold_exceedance_frequency.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 1 saved!")

## Plot 2: Duration Curves

Cumulative distribution showing how long heat stress persists.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Daytime heat duration curve
ax = axes[0]
summer_day_data = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

for state_name, state_id in FOCUS_STATES.items():
    state_data = compute_state_mean(summer_day_data, state_id, state_mask)
    sorted_data = np.sort(state_data.values[~np.isnan(state_data.values)])[::-1]
    exceedance_prob = np.arange(1, len(sorted_data) + 1) / len(sorted_data) * 100
    ax.plot(exceedance_prob, sorted_data, linewidth=2, label=state_name)

ax.set_xlabel('Exceedance Probability (%)')
ax.set_ylabel('Hours Above 30°C per Day')
ax.set_title('Duration Curve: Daytime Heat\n(Summer 2010-2023)', fontweight='bold', fontsize=13)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

# Nighttime recovery duration curve
ax = axes[1]
summer_night_data = ds_night['hours_above_24'].where(ds_night.time.dt.month.isin([6, 7, 8]), drop=True)

for state_name, state_id in FOCUS_STATES.items():
    state_data = compute_state_mean(summer_night_data, state_id, state_mask)
    sorted_data = np.sort(state_data.values[~np.isnan(state_data.values)])[::-1]
    exceedance_prob = np.arange(1, len(sorted_data) + 1) / len(sorted_data) * 100
    ax.plot(exceedance_prob, sorted_data, linewidth=2, label=state_name)

ax.set_xlabel('Exceedance Probability (%)')
ax.set_ylabel('Hours Above 24°C per Night')
ax.set_title('Duration Curve: Poor Nighttime Recovery\n(Summer 2010-2023)', fontweight='bold', fontsize=13)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_duration_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 2 saved!")

## Plot 3: Percentile Analysis

Compare different percentiles (50th, 90th, 95th, 99th) across states.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics_percentile = [
    ('hours_above_30', 'Daytime Hours > 30°C', ds_day),
    ('hours_above_24', 'Nighttime Hours > 24°C', ds_night),
    ('vpd_mean', 'Afternoon Mean VPD (kPa)', ds_vpd),
    ('hours_above_35', 'Daytime Hours > 35°C', ds_day)
]

percentiles = [50, 90, 95, 99]

for idx, (metric, title, ds) in enumerate(metrics_percentile):
    ax = axes[idx]
    
    # Filter for summer
    summer_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Compute percentiles for each state
    state_percentiles = {state: [] for state in FOCUS_STATES.keys()}
    
    for state_name, state_id in FOCUS_STATES.items():
        state_data = compute_state_mean(summer_data, state_id, state_mask)
        clean_data = state_data.values[~np.isnan(state_data.values)]
        
        for p in percentiles:
            state_percentiles[state_name].append(np.percentile(clean_data, p))
    
    # Plot grouped bars
    x = np.arange(len(percentiles))
    width = 0.2
    colors = plt.cm.tab10(range(len(FOCUS_STATES)))
    
    for i, (state_name, color) in enumerate(zip(FOCUS_STATES.keys(), colors)):
        offset = width * (i - len(FOCUS_STATES)/2 + 0.5)
        ax.bar(x + offset, state_percentiles[state_name], width, label=state_name, color=color)
    
    ax.set_xticks(x)
    ax.set_xticklabels([f'P{p}' for p in percentiles])
    ax.set_xlabel('Percentile')
    ax.set_ylabel(title)
    ax.set_title(f'{title}\nPercentile Comparison', fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_percentile_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 3 saved!")

## Plot 4: Risk Matrix (Frequency × Intensity)

2D heatmap showing combined risk of frequency and intensity.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 18))  # Updated to 4x4 for 14 states
axes = axes.flatten()

summer_day_data = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

# Define bins for frequency (% of days) and intensity (hours per day)
freq_bins = [0, 10, 25, 50, 75, 100]  # % of days
intensity_bins = [0, 3, 6, 9, 12]  # hours per day

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    state_data = compute_state_mean(summer_day_data, state_id, state_mask)
    clean_data = state_data.values[~np.isnan(state_data.values)]
    
    # Create risk matrix
    risk_matrix = np.zeros((len(intensity_bins)-1, len(freq_bins)-1))
    
    for i in range(len(intensity_bins)-1):
        for j in range(len(freq_bins)-1):
            # Count days in this intensity range
            in_intensity_range = (clean_data >= intensity_bins[i]) & (clean_data < intensity_bins[i+1])
            count = np.sum(in_intensity_range)
            freq_pct = (count / len(clean_data)) * 100
            
            # Check if frequency falls in this range
            if freq_bins[j] <= freq_pct < freq_bins[j+1]:
                risk_matrix[i, j] = 1
            
            # Store actual count for display
            risk_matrix[i, j] = count
    
    # Plot heatmap
    im = ax.imshow(risk_matrix, cmap='YlOrRd', aspect='auto', origin='lower')
    
    # Labels
    ax.set_xticks(range(len(freq_bins)-1))
    ax.set_xticklabels([f'{freq_bins[i]}-{freq_bins[i+1]}%' for i in range(len(freq_bins)-1)], rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(intensity_bins)-1))
    ax.set_yticklabels([f'{intensity_bins[i]}-{intensity_bins[i+1]}h' for i in range(len(intensity_bins)-1)], fontsize=8)
    ax.set_xlabel('Frequency (% of Summer Days)', fontsize=9)
    ax.set_ylabel('Intensity (Hours > 30°C per Day)', fontsize=9)
    ax.set_title(f'Risk Matrix: {state_name}\n(Summer 2010-2023)', fontweight='bold', fontsize=10)
    
    # Add text annotations
    for i in range(len(intensity_bins)-1):
        for j in range(len(freq_bins)-1):
            text = ax.text(j, i, f'{int(risk_matrix[i, j])}',
                          ha='center', va='center', color='black', fontsize=8)
    
    plt.colorbar(im, ax=ax, label='Number of Days')

# Hide unused subplots
for idx in range(len(FOCUS_STATES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_risk_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 4 saved!")

## Plot 5: Cumulative Stress Days

Accumulation of heat stress over the summer season.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))  # Updated to 4x4 for 14 states
axes = axes.flatten()

# Focus on a specific summer (2023)
year = 2023
summer_mask = (ds_day.time.dt.year == year) & (ds_day.time.dt.month.isin([6, 7, 8]))
summer_2023 = ds_day['hours_above_30'].where(summer_mask, drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    state_data = compute_state_mean(summer_2023, state_id, state_mask)
    
    # Define "stress day" as day with > 6 hours above 30°C
    stress_days = (state_data > 6).astype(int)
    cumulative = np.cumsum(stress_days.values)
    
    # Plot cumulative
    dates = pd.to_datetime(state_data.time.values)
    ax.plot(dates, cumulative, linewidth=2, color='darkred')
    ax.fill_between(dates, cumulative, alpha=0.3, color='red')
    
    ax.set_xlabel('Date', fontsize=9)
    ax.set_ylabel('Cumulative Stress Days', fontsize=9)
    ax.set_title(f'Cumulative Heat Stress Days: {state_name}\nSummer {year} (>6h above 30°C)',
                fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add final count
    final_count = cumulative[-1]
    ax.text(0.98, 0.02, f'Total: {final_count} days',
           transform=ax.transAxes, ha='right', va='bottom',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
           fontsize=9, fontweight='bold')

# Hide unused subplots
for idx in range(len(FOCUS_STATES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'05_cumulative_stress_days_{year}.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 5 saved!")

## Plot 6: Heat Wave Identification

Detect and characterize multi-day heat wave events.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))  # Updated to 4x4 for 14 states
axes = axes.flatten()

# Analyze all summers 2010-2023
summer_all = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    state_data = compute_state_mean(summer_all, state_id, state_mask)
    
    # Identify heat waves (3+ consecutive days with >8h above 30°C)
    heatwaves = identify_heatwaves(state_data, threshold=8, min_duration=3)
    
    # Analyze heat wave characteristics
    if heatwaves:
        durations = [hw['duration'] for hw in heatwaves]
        
        # Plot histogram of durations
        ax.hist(durations, bins=range(3, max(durations)+2), color='darkred',
               alpha=0.7, edgecolor='black')
        
        # Add statistics
        avg_duration = np.mean(durations)
        max_duration = np.max(durations)
        total_events = len(heatwaves)
        
        stats_text = f'Total events: {total_events}\n'
        stats_text += f'Avg duration: {avg_duration:.1f} days\n'
        stats_text += f'Max duration: {max_duration} days'
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               ha='right', va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
               fontsize=9)
    
    ax.set_xlabel('Heat Wave Duration (days)', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=9)
    ax.set_title(f'Heat Wave Characteristics: {state_name}\n(3+ days, >8h above 30°C, 2010-2023)',
                fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

# Hide unused subplots
for idx in range(len(FOCUS_STATES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_heatwave_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 6 saved!")

## Plot 7: Recovery Period Analysis

How long does it take to recover after extreme heat?

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))  # Updated to 4x4 for 14 states
axes = axes.flatten()

# Use nighttime recovery metric
summer_night = ds_night['hours_above_24'].where(ds_night.time.dt.month.isin([6, 7, 8]), drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    state_data = compute_state_mean(summer_night, state_id, state_mask)
    
    # Find extreme heat nights (>6 hours above 24°C)
    extreme_nights = state_data > 6
    
    # Find recovery periods (days until <2 hours above 24°C)
    recovery_times = []
    
    for i in range(len(extreme_nights)-1):
        if extreme_nights[i].values:
            # Look ahead for recovery
            for j in range(i+1, min(i+15, len(state_data))):
                if state_data[j].values < 2:
                    recovery_times.append(j - i)
                    break
    
    if recovery_times:
        # Plot histogram
        ax.hist(recovery_times, bins=range(1, max(recovery_times)+2),
               color='steelblue', alpha=0.7, edgecolor='black')
        
        # Add statistics
        avg_recovery = np.mean(recovery_times)
        median_recovery = np.median(recovery_times)
        
        stats_text = f'Mean: {avg_recovery:.1f} days\n'
        stats_text += f'Median: {median_recovery:.0f} days\n'
        stats_text += f'Sample size: {len(recovery_times)}'
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               ha='right', va='top',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
               fontsize=9)
    
    ax.set_xlabel('Days to Recovery', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=9)
    ax.set_title(f'Recovery Time After Extreme Heat: {state_name}\n(Time to <2h above 24°C)',
                fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

# Hide unused subplots
for idx in range(len(FOCUS_STATES), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_recovery_period_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 7 saved!")

## Plot 8: Consecutive Day Analysis

Distribution of consecutive days above different thresholds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

thresholds = [6, 8, 10, 12]  # hours per day above 30°C
summer_all = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

state_name = 'Texas'
state_id = FOCUS_STATES[state_name]
state_data = compute_state_mean(summer_all, state_id, state_mask)

for idx, threshold in enumerate(thresholds):
    ax = axes[idx]
    
    # Find all consecutive sequences
    above_thresh = (state_data > threshold).values.astype(int)
    
    consecutive_lengths = []
    current_length = 0
    
    for val in above_thresh:
        if val == 1:
            current_length += 1
        else:
            if current_length > 0:
                consecutive_lengths.append(current_length)
            current_length = 0
    
    # Don't forget last sequence
    if current_length > 0:
        consecutive_lengths.append(current_length)
    
    if consecutive_lengths:
        # Plot histogram
        max_len = min(max(consecutive_lengths), 30)  # Cap at 30 days for readability
        ax.hist(consecutive_lengths, bins=range(1, max_len+2),
               color=plt.cm.Reds(0.3 + idx*0.15), alpha=0.7, edgecolor='black')
        
        # Statistics
        avg_length = np.mean(consecutive_lengths)
        max_length = np.max(consecutive_lengths)
        total_sequences = len(consecutive_lengths)
        
        stats_text = f'Total sequences: {total_sequences}\n'
        stats_text += f'Average length: {avg_length:.1f} days\n'
        stats_text += f'Longest: {max_length} days'
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               ha='right', va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
               fontsize=9)
    
    ax.set_xlabel('Consecutive Days')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Consecutive Days > {threshold}h Above 30°C\n{state_name} (2010-2023)',
                fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xlim(0, max_len+1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_consecutive_day_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 8 saved!")
print("\n=== Threshold and Duration Analysis Complete ===")
print(f"8 plots saved to {OUTPUT_DIR.resolve()}")

## Summary

This notebook generated 8 comprehensive threshold and duration analysis plots:

1. **Threshold Exceedance Frequency** - How often different intensity levels occur
2. **Duration Curves** - Cumulative distribution of heat stress persistence
3. **Percentile Analysis** - Compare extreme values (P50, P90, P95, P99)
4. **Risk Matrix** - Combined frequency × intensity risk assessment
5. **Cumulative Stress Days** - Seasonal accumulation of heat stress
6. **Heat Wave Characteristics** - Multi-day extreme event analysis
7. **Recovery Period Analysis** - Time required to recover from extreme heat
8. **Consecutive Day Analysis** - Persistence of heat stress conditions

**Key Insights:**
- Texas experiences the most frequent threshold exceedances
- Heat waves average 5-7 days in Southern Plains states
- Recovery after extreme heat typically takes 2-4 days
- Consecutive heat stress days can exceed 2 weeks in extreme years
- 95th percentile conditions occur ~5% of summer days but pose significant risk